# Function Calling
- Function Calling은 OpenAI API에서 **특정 함수 정의(JSON schema 기반)** 를 주고, 모델이 해당 **함수를 호출하는 형식의 응답** 을 생성하게 하는 기능입니다.
- 챗봇, 에이전트(Agent), 도구(Tool) 사용 등 다양한 자동화 및 플러그인 연동에 활용됩니다.

- 지원 모델 gpt-3-turbo, gpt-4, gpt-4-turbo  (2025.1 현재)

- 모델이 우리의 코드를 호출하도록 해서 우리의 함수들을 모델이 호출할수 있도록 하거나
- 모델이 우리가 원하는 특정 모양과 형식의 output 을 갖도록 강제할수 있다.

- https://platform.openai.com/docs/guides/function-calling?api-mode=chat

![](https://cdn.thenewstack.io/media/2024/05/5026cffa-rag_function_call-1024x835.jpeg)

출처: https://thenewstack.io/a-comprehensive-guide-to-function-calling-in-llms/?utm_source=chatgpt.com


In [17]:
import os 

from dotenv import load_dotenv
load_dotenv()

print(f'OPENAI_API_KEY={os.getenv("OPENAI_API_KEY")[:20]}...')

OPENAI_API_KEY=sk-proj-DPrT3N-1Ilp9...


In [18]:
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate

In [3]:
llm = ChatOpenAI(
    temperature=0.1,
)

prompt = PromptTemplate.from_template("Who is the weather in {city}")  
chain = prompt | llm

response = chain.invoke({
    "city": "rome"
})

print(response.content)


I'm sorry, I am not able to provide real-time weather updates. I recommend checking a reliable weather website or app for the most up-to-date information on the weather in Rome.


In [4]:
# 사전합습모델은 실시간으로 날씨를 가져올수 없다.
# 하지만!  이제 함수가 있다고 생각해보자.  실시간 데이터를 가져올 수 잇는 함수 말이다!
#  함수에 필요한 정보는 아마 '장소' 정보 뿐 일거다.  (위도, 경도..)

# ↓여기에 함수가 있다고 가정하자

In [19]:
import httpx

In [20]:
# 위도, 경도 가 주어지면 실시간 날씨 정보를 리턴하는 함수
def get_weather(lon, lat):
    print(f'🔵 get_weather({lon}, {lat}) 호출')
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    response = httpx.get(url)
    response.raise_for_status()  # http 통신 응답 에러시 예외 발생
    return response.json()

In [21]:
coord_seoul = {"lat": 37.56654, "lon": 126.97797}

In [22]:
get_weather(**coord_seoul)

🔵 get_weather(126.97797, 37.56654) 호출


{'latitude': 37.55,
 'longitude': 127.0,
 'generationtime_ms': 0.13267993927001953,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 34.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2026-06-24T07:15',
  'interval': 900,
  'temperature': 28.8,
  'windspeed': 3.4,
  'winddirection': 108,
  'is_day': 1,
  'weathercode': 1}}

In [ ]:
# AI 에게 우리에겐 이러한 함수가 있다 라는 것을 알려준다
# -> JSON schema 를 만들어 알려준다.

# 함수의 스키마

In [23]:
# 함수의 작동방식과 필요한 것을 설명해주는 것이 스키마다
# ↓ 아래와 같은 방법으로 함수의 형태와 필요한 데이터를 설명해주고 있다.
function_schema = {
    "name": "get_weather",  # 함수의 이름
    # description : 함수가 무슨 일을 하는지 기술
    # ↓'위도와 경도를 받아서 특정장소의 날씨정보를 가져오는 함수"
    "description": "function that takes longitude and latitude to find the weather of a place", 

    # 파라미터 기술
    "parameters": {
        "type": "object",
        "properties": {
            "lon": {"type": "number", "description": "The longitude coordinate"},
            "lat": {"type": "number", "description": "The latitude coordinate"},
        },
    },

    # 필수사항 기술
    "required": ["lon", "lat"],  
}


## LLM 에 함수 스키마 전달하기

In [24]:
llm = ChatOpenAI(
    temperature=0.1
).bind(    # LLM 에 전달한 추가 인자들 
    functions=[
        function_schema,   # <-  함수의 schema 전달
    ],

    
)

In [27]:
chain = prompt | llm

response = chain.invoke({
    "city": "rome"
})

response

Task was destroyed but it is pending!
task: <Task pending name='Task-188' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/Dropbox/K16/PyWork/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-189' coro=<Kernel.shell_main() running at /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/Dropbox/K16/PyWork/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/Dropbox/K16/PyWork/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/Dropbox/K16/PyWork/.venv/lib/python3.12/site-packages/langchain_core/load/_validation.py:122: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  secrets[alias] = secrets[key]
Task was destroyed but it is pending!
task: <Task pending name='

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"lon":12.4964,"lat":41.9028}', 'name': 'get_weather'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 74, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DuCK5x1HnQcwRp84AgYXqlxraQvBe', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--019ef887-c5d5-7e51-80f7-b7c4aa69fd20-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 24, 'total_tokens': 98, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
# response.content 에는 내용이 없다.

In [25]:
"""
↑ 함 들여다보자.

AIMessage(
    content='',   # <-- 응답은 받았지만 content 는 없다.  대신! 함수가 호출된것을 볼수 있다!
    additional_kwargs={
        # ↓ 함수가 호출되었다!  
        # 모델은 get_weather 라는 함수를 호출해 줬으면 해!  라고 말하고 있는거다.
        # 그리고, "네가 경도와 위도에 대한 argument 를 제공해 줬으면 해" 라고 하고 있는 거다.        
        # rome 의 lon 과 lat 값을 전달해 주었다!  오오오오!
        'function_call': {'arguments': '{"lon":"12.4964","lat":"41.9028"}', 'name': 'get_weather'}, 
        'refusal': None
    }, 
"""
None

In [28]:
response.additional_kwargs

{'function_call': {'arguments': '{"lon":12.4964,"lat":41.9028}',
  'name': 'get_weather'},
 'refusal': None}

In [29]:
response.additional_kwargs['function_call']

{'arguments': '{"lon":12.4964,"lat":41.9028}', 'name': 'get_weather'}

In [32]:
json_str = response.additional_kwargs['function_call']['arguments']
json_str

'{"lon":12.4964,"lat":41.9028}'

In [33]:
import json
r = json.loads(json_str)
r

{'lon': 12.4964, 'lat': 41.9028}

In [34]:
get_weather(**r)

🔵 get_weather(12.4964, 41.9028) 호출


{'latitude': 41.875,
 'longitude': 12.5,
 'generationtime_ms': 0.05829334259033203,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 58.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2026-06-24T07:30',
  'interval': 900,
  'temperature': 28.3,
  'windspeed': 4.8,
  'winddirection': 27,
  'is_day': 1,
  'weathercode': 0}}

# QuizGPT의 퀴즈 생성 함수 스키마

In [36]:
function_schema = {
    # '퀴즈 생성하기' 스키마
    "name": "create_quiz",  # 실제로는 존재하지 않는 함수다.
    "description": "function that takes a list of questions and answers and returns a quiz",

    # ↓ 우라기 원하는 응답의 스키마는
 
# 바로 이 형태를 정의하는 거다. 
# {
#   "questions":[
#      0:{
#        "question":"What … in the story?"
#        "answers":[
#           0:{
#             "answer":"John"
#             "correct":false
#           }
#           ... 
#        ]
#      }
#      ...
# } 

    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "question": {
                            "type": "string",
                        },
                        "answers": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "answer": {
                                        "type": "string"
                                    },
                                    "correct": {
                                        "type": "boolean"
                                    },
                                },
                                "required": ["answer", "correct"],
                            },
                        },
                    },
                    "required": ["question", "answers"],
                },
            },
        },
        "required": ["questions"],
    },
    
}


In [38]:
llm = ChatOpenAI(
    temperature=0.1
).bind(
    functions=[function_schema],

    # function_call="auto"  # "auto"  모델이 우리 함수를 사용할 수도 있고, 않을수도 있다
    function_call = {  # 모델이 강제로 함수를 사용하도록 할수도 있다.
        "name": "create_quiz",  
    }
)

prompt = PromptTemplate.from_template("Make a quiz about {city}")
chain = prompt | llm
response = chain.invoke({"city": "roma"})

response = response.additional_kwargs['function_call']['arguments']

json.loads(response)


{'questions': [{'question': 'What is the capital city of Italy?',
   'answers': [{'answer': 'Rome', 'correct': True},
    {'answer': 'Milan', 'correct': False},
    {'answer': 'Florence', 'correct': False},
    {'answer': 'Venice', 'correct': False}]},
  {'question': 'Who was the famous leader of the Roman Empire known for his military conquests?',
   'answers': [{'answer': 'Julius Caesar', 'correct': True},
    {'answer': 'Alexander the Great', 'correct': False},
    {'answer': 'Napoleon Bonaparte', 'correct': False},
    {'answer': 'Genghis Khan', 'correct': False}]},
  {'question': 'What is the name of the famous ancient Roman amphitheater that hosted gladiatorial contests?',
   'answers': [{'answer': 'Colosseum', 'correct': True},
    {'answer': 'Pantheon', 'correct': False},
    {'answer': 'Forum', 'correct': False},
    {'answer': 'Circus Maximus', 'correct': False}]}]}